# 02 · Silver : nettoyage

Dédoublonne, normalise, type les colonnes et calcule les délais et le coût total.
Les bons qu'on ne peut pas corriger sans inventer une valeur vont dans `silver_quarantaine`, avec leur motif.

| Défaut | Traitement |
|---|---|
| Doublons exacts | Retirés |
| Catégories et codes de pavillon mal écrits | Normalisés |
| Dates `JJ/MM/AAAA HH:MM` | Lues avec leur propre format |
| Coût de pièces manquant | Compté comme 0 $ dans le total et signalé par `cout_pieces_manquant` |
| Technicien manquant sur un bon fermé | Remplacé par `INCONNU` |
| Fermeture avant création, coût négatif, heures aberrantes | Quarantaine |

In [ ]:
FORMAT_TABLE = "delta"
HEURES_MAX_PAR_BON = 200

In [ ]:
from pyspark.sql import functions as F

CATEGORIES = {
    "plomberie": "Plomberie", "plomb.": "Plomberie",
    "électricité": "Électricité", "electricite": "Électricité", "elec": "Électricité",
    "cvc": "CVC", "hvac": "CVC", "c.v.c.": "CVC",
    "dégât d'eau": "Dégât d'eau", "dégâts d'eau": "Dégât d'eau",
    "degat d'eau": "Dégât d'eau", "degat eau": "Dégât d'eau",
}


def ecrire(df, nom):
    df.write.format(FORMAT_TABLE).mode("overwrite").option("overwriteSchema", "true").saveAsTable(nom)
    print(f"{nom} : {spark.table(nom).count()} lignes")


def lire_date(nom):
    """Chaque ligne est lue avec son format, plutôt que d'essayer les deux à l'aveugle."""
    texte = F.col(nom)
    return (
        F.when(texte.rlike(r"^\d{2}/"), F.to_timestamp(texte, "dd/MM/yyyy HH:mm"))
        .otherwise(F.to_timestamp(texte, "yyyy-MM-dd HH:mm:ss"))
    )


def heures_entre(debut, fin):
    return (F.col(fin).cast("long") - F.col(debut).cast("long")) / 3600

## Tables de référence

In [ ]:
pavillons = spark.table("bronze_pavillons").select(
    "pavillon_id", "nom", "vocation",
    F.col("annee_construction").cast("int").alias("annee_construction"),
    F.col("superficie_m2").cast("int").alias("superficie_m2"),
    F.col("nb_etages").cast("int").alias("nb_etages"),
)
locaux = spark.table("bronze_locaux").select(
    "local_id", "pavillon_id",
    F.col("etage").cast("int").alias("etage"),
    "type_local",
    F.col("criticite").cast("int").alias("criticite"),
)
techniciens = spark.table("bronze_techniciens").select(
    "technicien_id", "metier", "quart", "statut_emploi",
    F.col("taux_horaire").cast("double").alias("taux_horaire"),
)
priorites = spark.table("bronze_priorites").select(
    "priorite", "libelle",
    F.col("cible_fermeture_heures").cast("int").alias("cible_fermeture_heures"),
)

ecrire(pavillons, "silver_pavillons")
ecrire(locaux, "silver_locaux")
ecrire(techniciens, "silver_techniciens")
ecrire(priorites, "silver_priorites")

## Doublons

La comparaison ignore les colonnes techniques de bronze, qui commencent par `_`.

In [ ]:
bronze = spark.table("bronze_bons_de_travail")
colonnes_metier = [c for c in bronze.columns if not c.startswith("_")]
dedoublonnes = bronze.select(colonnes_metier).dropDuplicates()
print(f"{bronze.count() - dedoublonnes.count()} doublons exacts retirés")

## Normalisation et typage

In [ ]:
correspondance = F.create_map(*[F.lit(v) for paire in CATEGORIES.items() for v in paire])
colonnes_dates = ["date_creation", "date_assignation", "date_debut_travaux", "date_fermeture"]
colonnes_nombres = ["heures_main_oeuvre", "cout_main_oeuvre", "cout_pieces", "cout_sous_traitance"]

types = (
    dedoublonnes
    .withColumn("categorie", correspondance[F.lower(F.trim("categorie"))])
    .withColumn("pavillon_id", F.regexp_replace(F.upper(F.trim("pavillon_id")), " ", "-"))
    .withColumn("etage", F.col("etage").cast("int"))
)
for colonne in colonnes_dates:
    types = types.withColumn(colonne, lire_date(colonne))
for colonne in colonnes_nombres:
    types = types.withColumn(colonne, F.col(colonne).cast("double"))

ferme = F.col("statut") == "Fermé"
enrichis = (
    types
    .withColumn("cout_pieces_manquant", ferme & F.col("cout_pieces").isNull())
    .withColumn("technicien_id", F.when(ferme, F.coalesce("technicien_id", F.lit("INCONNU"))).otherwise(F.col("technicien_id")))
    .withColumn(
        "cout_total",
        F.when(ferme, F.round(
            F.col("cout_main_oeuvre") + F.coalesce("cout_pieces", F.lit(0.0)) + F.col("cout_sous_traitance"), 2,
        )),
    )
    .withColumn("delai_assignation_heures", heures_entre("date_creation", "date_assignation"))
    .withColumn("delai_fermeture_heures", heures_entre("date_creation", "date_fermeture"))
)

## Quarantaine

Un bon peut avoir plusieurs motifs. `F.when` sans `otherwise` donne `null`, retiré ensuite par `array_compact`.

In [ ]:
ids_pavillons = [r.pavillon_id for r in pavillons.collect()]

regles = {
    "date_creation_illisible": F.col("date_creation").isNull(),
    "categorie_inconnue": F.col("categorie").isNull(),
    "pavillon_inconnu": ~F.col("pavillon_id").isin(ids_pavillons),
    "fermeture_avant_creation": F.col("date_fermeture") < F.col("date_creation"),
    "fermeture_manquante": ferme & F.col("date_fermeture").isNull(),
    "cout_negatif": F.col("cout_pieces") < 0,
    "heures_aberrantes": F.col("heures_main_oeuvre") > HEURES_MAX_PAR_BON,
}
motifs = F.array_compact(F.array(*[F.when(condition, F.lit(motif)) for motif, condition in regles.items()]))

evalues = enrichis.withColumn("motifs", motifs).cache()
valides = evalues.filter(F.size("motifs") == 0).drop("motifs")
quarantaine = (
    evalues.filter(F.size("motifs") > 0)
    .withColumn("motifs", F.concat_ws(", ", "motifs"))
    .withColumn("date_quarantaine", F.current_timestamp())
)

ecrire(valides, "silver_bons_de_travail")
ecrire(quarantaine, "silver_quarantaine")
display(spark.table("silver_quarantaine").groupBy("motifs").count().orderBy(F.desc("count")))

## Contrôles

Le notebook échoue si une règle n'est pas respectée : dans un pipeline, la couche gold ne sera pas recalculée.

In [ ]:
silver = spark.table("silver_bons_de_travail")
ids_doubles = silver.groupBy("bon_id").count().filter("count > 1").count()
locaux_inconnus = silver.join(spark.table("silver_locaux"), "local_id", "left_anti").count()
techniciens_inconnus = (
    silver.filter("technicien_id IS NOT NULL AND technicien_id <> 'INCONNU'")
    .join(spark.table("silver_techniciens"), "technicien_id", "left_anti").count()
)
priorites_inconnues = silver.join(spark.table("silver_priorites"), "priorite", "left_anti").count()
total = silver.count() + spark.table("silver_quarantaine").count()

assert ids_doubles == 0, f"{ids_doubles} bon_id en double"
assert locaux_inconnus == 0, f"{locaux_inconnus} bons avec un local inconnu"
assert techniciens_inconnus == 0, f"{techniciens_inconnus} bons avec un technicien inconnu"
assert priorites_inconnues == 0, f"{priorites_inconnues} bons avec une priorité inconnue"
assert total == dedoublonnes.count(), "des bons ont été perdus entre bronze et silver"
print("Contrôles réussis")
evalues.unpersist()